# Entrenamiento del modelo Random Forest
Este notebook es ejecutado por Airflow via la API de Jupyter.
Lee TODOS los batches acumulados en PostgreSQL (no solo el actual),
entrena el modelo y guarda un bundle con modelo + encoders en MinIO.
Asi el modelo mejora progresivamente con cada nueva ejecucion del DAG.

In [ ]:
import os
import io
import pickle
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import psycopg2
import boto3
from botocore.client import Config
from datetime import datetime

print('Librerias cargadas correctamente')

In [ ]:
# Parametros inyectados por el DAG de Airflow via variables de entorno
BATCH_NUMBER = int(os.environ.get('BATCH_NUMBER', 1))
GROUP_NUMBER = int(os.environ.get('GROUP_NUMBER', 4))

# Conexion a PostgreSQL
PG_HOST     = os.environ.get('PG_HOST', 'postgres')
PG_PORT     = os.environ.get('PG_PORT', '5432')
PG_DB       = os.environ.get('PG_DB', 'airflow')
PG_USER     = os.environ.get('PG_USER', 'airflow')
PG_PASSWORD = os.environ.get('PG_PASSWORD', 'airflow')

# Conexion a MinIO
MINIO_ENDPOINT   = os.environ.get('MINIO_ENDPOINT', 'http://minio:9000')
MINIO_ACCESS_KEY = os.environ.get('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.environ.get('MINIO_SECRET_KEY', 'minioadmin')
MINIO_BUCKET     = 'models'

print(f'Parametros: batch_actual={BATCH_NUMBER}, group={GROUP_NUMBER}')

In [ ]:
# Cargar TODOS los batches acumulados hasta el momento
# Sin filtrar por batch_number para acumular todos los datos disponibles
print(f'Conectando a PostgreSQL en {PG_HOST}:{PG_PORT}...')

conn = psycopg2.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DB,
    user=PG_USER,
    password=PG_PASSWORD
)

query = """
    SELECT
        elevation, aspect, slope,
        horizontal_distance_to_hydrology, vertical_distance_to_hydrology,
        horizontal_distance_to_roadways,
        hillshade_9am, hillshade_noon, hillshade_3pm,
        horizontal_distance_to_fire_points,
        wilderness_area_encoded, soil_type_encoded,
        cover_type
    FROM public.training_data_processed
    WHERE group_number = %s
    AND cover_type BETWEEN 1 AND 7
"""

df = pd.read_sql(query, conn, params=(GROUP_NUMBER,))

query_text = """
    SELECT DISTINCT wilderness_area, soil_type
    FROM public.training_data
    WHERE group_number = %s
    AND cover_type BETWEEN 1 AND 7
"""
df_text = pd.read_sql(query_text, conn, params=(GROUP_NUMBER,))

df_batches = pd.read_sql(
    f"SELECT DISTINCT batch_number FROM training_data_processed WHERE group_number = {GROUP_NUMBER} ORDER BY batch_number",
    conn
)
conn.close()

batches_usados = list(df_batches['batch_number'])
print(f'Batches usados para entrenamiento: {batches_usados}')
print(f'Total filas: {len(df)}')
print(df.head())

In [ ]:
# Reconstruir los encoders a partir de todos los valores de texto vistos
le_wilderness = LabelEncoder()
le_soil       = LabelEncoder()

le_wilderness.fit(df_text['wilderness_area'].values)
le_soil.fit(df_text['soil_type'].values)

print(f'Wilderness areas en el encoder: {list(le_wilderness.classes_)}')
print(f'Soil types en el encoder: {len(le_soil.classes_)} tipos')

In [ ]:
# Preparar datos para entrenamiento
feature_cols = [
    'elevation', 'aspect', 'slope',
    'horizontal_distance_to_hydrology', 'vertical_distance_to_hydrology',
    'horizontal_distance_to_roadways',
    'hillshade_9am', 'hillshade_noon', 'hillshade_3pm',
    'horizontal_distance_to_fire_points',
    'wilderness_area_encoded', 'soil_type_encoded'
]

X = df[feature_cols].values
y = df['cover_type'].values

print(f'Distribucion de clases:')
for clase in sorted(set(y)):
    print(f'  cover_type={clase}: {sum(y==clase)} filas')

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y if len(np.unique(y)) > 1 else None
)

print(f'Train: {len(X_train)} filas | Test: {len(X_test)} filas')

In [ ]:
# Entrenar el modelo Random Forest
#
# PROBLEMA DETECTADO: el modelo predecia siempre la clase mas frecuente
# (cover_type=1 con ~10000 filas vs cover_type=3 con ~240 filas).
# Esto se llama sesgo por desbalance de clases: el modelo aprende que
# predecir siempre '1' le da un accuracy alto, ignorando las clases
# minoritarias como 3, 4 o 6.
#
# SOLUCION: class_weight='balanced' calcula automaticamente un peso
# inversamente proporcional a la frecuencia de cada clase:
#   peso = total_filas / (num_clases * filas_de_esa_clase)
#
# Ejemplo con los datos actuales:
#   cover_type=1: peso bajo  (muchas filas, errores poco penalizados)
#   cover_type=3: peso alto  (pocas filas, errores muy penalizados)
#
# El modelo aprende a prestar igual atencion a todas las clases,
# produciendo predicciones mas variadas y correctas.
# El accuracy general puede bajar un poco, pero las predicciones
# seran mas justas para todas las clases.
print('Entrenando Random Forest con class_weight=balanced...')

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'   # <- corrige el sesgo hacia clases mayoritarias
)
model.fit(X_train, y_train)

y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Entrenamiento completado')
print(f'Accuracy en test: {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Clases predichas en test: {sorted(set(y_pred))}')

In [ ]:
# Guardar el bundle en MinIO
print(f'Conectando a MinIO en {MINIO_ENDPOINT}...')

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

existing = [b['Name'] for b in s3.list_buckets().get('Buckets', [])]
if MINIO_BUCKET not in existing:
    s3.create_bucket(Bucket=MINIO_BUCKET)
    print(f"Bucket '{MINIO_BUCKET}' creado")

bundle = {
    'model':         model,
    'le_wilderness': le_wilderness,
    'le_soil':       le_soil
}

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
object_name = f'group_{GROUP_NUMBER}/batch_{BATCH_NUMBER}_acc_{round(accuracy, 4)}_{timestamp}.pkl'

bundle_bytes = pickle.dumps(bundle)
s3.upload_fileobj(
    io.BytesIO(bundle_bytes),
    MINIO_BUCKET,
    object_name
)

print(f'Bundle guardado en MinIO: s3://{MINIO_BUCKET}/{object_name}')
print(f'Tamano: {len(bundle_bytes) / 1024:.1f} KB')
print(f'Batches usados: {batches_usados}')
print(f'Total filas de entrenamiento: {len(X_train)}')
print('Notebook completado exitosamente')